# 黑马头条项目｜第12天：项目部署与持续集成 Jenkins + Git + Docker

> 学习目标：理解持续集成与软件迭代，掌握 Jenkins 工具/凭据配置、Maven 多模块构建、Nacos 多环境配置、Docker 镜像与私有仓库、远程部署、构建触发器和部署验证。

本篇依据百度网盘 `day12-项目部署 持续集成发布Jenkins+Git+Docker/讲义/项目部署_持续集成.md` 完整网页预览整理。已查看资料目录文件名，包含 SQL脚本目录、`.gitignore`、Maven安装包、CentOS压缩包和本地依赖仓库压缩包；未读取这些文件的正文或解压安装包，`.gitignore`点击后未显示正文。本篇没有读取参考代码、PPT、视频和图片资源包。

所有服务器地址、仓库地址、用户凭据和构建Token均用占位符或角色名称表示。以下命令、Dockerfile与Pipeline仅作为笔记示意，未执行系统安装、权限修改、Git上传、镜像推送、远程登录、容器替换、SQL导入或真实构建部署。

“课程约定”记录实际讲义；“工程补充”“修正示意”解释兼容性、正确性和发布边界，不代表课程参考工程已经实现这些改进。原图没有嵌入，流程以文字与表格说明。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 持续集成、持续交付与部署

讲义把持续集成（Continuous Integration，CI）定义为频繁地把代码集成到主干，由自动化过程完成检出、编译、测试、结果记录和统计。

三个组成要素：版本控制仓库、自动构建过程、持续集成服务器。本课分别使用 Git、JDK/Maven/Docker 和 Jenkins。

| 概念 | 核心结果 | 工程补充的边界 |
|---|---|---|
| CI：持续集成 | 每次集成能自动构建、测试并反馈 | 绿色构建不等于生产环境已部署 |
| 持续交付 | 有一个通过验证、可部署的版本 | 发布前可保留人工审批 |
| 持续部署 | 合格版本自动进入目标环境 | 需明确环境与发布策略 |

~~~text
代码提交 → 检出固定版本 → 编译/测试 → 打包 → 构建镜像
                                                 ↓
                             镜像仓库 → 测试环境验证 → 发布
                                                 ↓
                                        观测、反馈、回滚
~~~

好处是尽早发现集成问题、减少重复工作、降低发布风险并提供可追踪制品。跳过测试的自动脚本不能完整替代CI质量检查。


## 2. 软件生命周期与敏捷迭代

讲义介绍 SDLC（Software Development Life Cycle）：需求分析、设计、实现、测试和持续演进，部署贯穿交付阶段。

| 模型/概念 | 说明 |
|---|---|
| 瀑布 | 按阶段推进，检查点明确；反馈晚、需求变更代价高 |
| 迭代 | 把大周期拆成多个小周期，每次重新经过完整开发流程 |
| 增量 | 每次交付一个用户可感知的完整功能 |

以黑马头条为例：先交付文章浏览，再交付自媒体发布/审核，再交付搜索、行为和热点推荐。每次增量仍要设计、开发、测试、部署，而不是“敏捷就不做文档、不做验证”。

CI把每次小变更自动验证，使团队更早看到可交付结果。开发模式应结合需求稳定程度和项目约束选择；讲义对瀑布的批评不等于它在所有项目里都不可用。


## 3. Jenkins、Git、Maven与Docker的分工

| 工具/角色 | 本课职责 |
|---|---|
| Git仓库 | 保存源码、版本和分支，作为构建输入 |
| Jenkins | 检出源码、编排工具步骤、保存日志与构建结果 |
| JDK/Maven | 编译Java、多模块依赖管理、测试、生成可执行JAR |
| Docker | 将JAR与运行环境打成镜像，启动容器 |
| Registry | 集中保存镜像，供目标服务器拉取 |
| SSH | 连接目标服务器执行部署步骤 |
| Nacos | 服务发现和环境配置 |
| Nginx/APP网关 | 将页面请求转入部署后的服务链路 |

~~~text
开发者 → Git
          ↓ Jenkins构建环境：检出 → Maven → Docker镜像
          ↓ Registry
目标运行环境：拉取同一镜像 → 加载对应Nacos配置 → 启动服务
          ↓
Nginx → APP网关 → 用户/文章/行为等微服务
~~~

构建机器、镜像仓库和运行机器可以分开。本课先演示构建机本地启动，再演示远程部署，笔记使用角色名区分，不复制示例内网IP。


## 4. Jenkins安装内容与课程版本

讲义展示Linux YUM、RPM方式安装Jenkins、服务用户与监听端口配置、Java路径、首次解锁、推荐插件安装、管理员创建和Jenkins URL设置。

示例版本包括 Jenkins RPM `2.190.1`、Git `1.8.3.1`、Maven `3.6.1`，项目编译目标是Java8。这些属于课程历史版本，不代表当前安装建议；新环境应按Jenkins支持矩阵与项目依赖选择版本。

需要区分两套Java：Jenkins自身运行所需JDK与业务项目编译/运行JDK，二者不必相同。不能因为项目是Java8，就假设当前Jenkins也能用Java8启动。

初始化要点：

1. Jenkins服务可正常启动，Java路径正确；
2. 用本地初始解锁凭据完成初始化，不把凭据贴进日志或笔记；
3. 安装所需插件、配置管理员和访问URL；
4. 校验服务用户对工作区、缓存和日志目录的权限；
5. 明确构建节点可以访问Git、依赖仓库、Docker和Registry。

工程补充：无需照搬课程中的root运行、放宽所有权限、关闭证书校验或旧安装源。控制器管理与构建任务可分离到受控Agent，减少构建代码对Jenkins主机的影响。


## 5. 插件、工具配置与凭据

讲义列出的插件：

- Maven Integration plugin；
- Docker plugin；
- GitLab Plugin；
- Publish Over SSH；
- SSH远程脚本执行插件。

它们用于课程的任务类型、源码接入和发布流程，不表示任何Jenkins项目都必须全部安装。插件兼容性、维护状态和实际功能需按环境确认；图片中的额外插件未读取具体名称，不编造列表。

全局工具配置要指定JDK、Maven、Docker及Maven settings路径。Git、JDK、Maven、Docker客户端和Docker守护进程是不同组件，工具路径正确也不代表运行权限与网络正常。

Git仓库、Registry和SSH目标使用分别授权的凭据，放入Jenkins Credentials并由ID引用。不要把密码写进Git URL、Dockerfile、POM、Shell正文或日志；添加凭据与测试连接也不等于已经部署业务服务。

现代版本中页面名称可能与讲义的“系统管理/全局工具配置”不同，按实际界面对应，不按历史截图位置硬套。


## 6. Git检出与构建版本

讲义通过Jenkins任务配置Git仓库和访问凭据，将源码拉到工作区。可复现构建还应明确仓库、分支、提交SHA和工作区路径。

~~~text
Git仓库 → 选定commit → Jenkins工作区 → 对应模块pom.xml
~~~

工程补充：

- 每次构建记录commit与构建编号，制品能够追溯到源码；
- 分支选择只是定位方式，最终发布应记录实际commit；
- 不向仓库提交密码、令牌、私钥、本地仓库或编译输出；
- `.gitignore`只影响未跟踪文件，不会自动移除已提交的秘密或JAR；
- `.dockerignore`控制镜像构建上下文，与`.gitignore`不同。

配套目录中的`.gitignore`未显示正文，故本篇不把常见忽略规则写成课程原文件内容。如果Dockerfile需要`target/<finalName>.jar`，也不能把整个target目录从构建上下文排除而没有另外传入制品。

Jenkins可自动检出代码；部署过程中不应再随意从远端重新拉取一个更晚版本，把验证过的commit换成另一份输入。


## 7. Maven多模块与基础依赖

讲义先构建父工程/基础依赖，将它们install到构建服务器的本地Maven仓库，再单独构建微服务，例如user或article。

常见Maven生命周期：

| 阶段/选项 | 含义 |
|---|---|
| clean | 清理上次构建输出 |
| test | 编译并执行单元测试 |
| package | 生成JAR等制品 |
| verify | 执行配置的验证阶段 |
| install | 把当前制品安装到本地Maven仓库 |
| `-f` | 指定POM文件 |
| `-pl`、`-am` | 选择模块并同时构建所需项目，属补充方案 |

父POM、common/model/feign等依赖如果只存在开发者本机，Jenkins单模块构建会出现依赖缺失。课程基础install步骤用于解决这个问题，但本地旧SNAPSHOT也可能使两个构建混用不同版本。

工程补充示意：在根工程按模块与依赖一起构建，路径按真实项目调整。

~~~bash
mvn -B clean verify -pl <MODULE_PATH> -am
~~~

示意含占位符，未执行。配套`repository_new.zip`是本地依赖仓库材料，不应当作可长期替代受控依赖仓库的部署制品来源。


## 8. 课程Maven镜像构建命令

讲义的user模块命令（项目相对路径按实际工作区核对）：

~~~text
clean install -Dmaven.test.skip=true dockerfile:build
  -f heima-leadnews/heima-leadnews-service/heima-leadnews-user/pom.xml
~~~

article模块使用相同方式替换POM路径。这里是Jenkins Maven Goals展示，直接在终端运行还需Maven调用入口。

解释：

- `clean install`生成并安装制品；
- `-Dmaven.test.skip=true`通常跳过测试编译和运行；`-DskipTests`通常只跳过运行，二者不同；
- `dockerfile:build`调用镜像构建插件，**构建的是镜像，不是运行中的容器**；
- `-f`指向POM，不是JAR或Dockerfile。

讲义跳过测试便于演示发布，但CI流程应有实际的编译、测试和验证关口。没有测试结果不能把“install成功”报告为“所有业务已通过测试”。

`-f`相对路径依赖Jenkins检出目录；如果仓库已检出在根目录，再多加一层`heima-leadnews`会找不到POM。


## 9. 多环境配置与Nacos Data ID

讲义划分Development、Test、Production环境，用profile选择不同数据库和其他连接。示例在微服务`bootstrap.yml`中设置应用名、Nacos发现/配置地址、扩展名以及`spring.profiles.active`。

占位配置示意：

~~~yaml
spring:
  application:
    name: leadnews-user
  profiles:
    active: dev
  cloud:
    nacos:
      discovery:
        server-addr: <NACOS_DISCOVERY_ADDRESS>
      config:
        server-addr: <NACOS_CONFIG_ADDRESS>
        file-extension: yml
~~~

Data ID约定：

~~~text
<prefix>-<profile>.<file-extension>
例如：leadnews-user-dev.yml / leadnews-user-test.yml / leadnews-user-prod.yml
profile为空：<prefix>.<file-extension>
~~~

prefix默认应用名，可通过`spring.cloud.nacos.config.prefix`指定。原说明文字有`spring.profile.active`和`file-exetension`拼写，而配置示例使用`spring.profiles.active`、`file-extension`；实际以正确配置项及当前版本为准。

工程补充：profile不是独立权限边界，还应核对namespace/group、Nacos地址、凭据和依赖资源。不同Spring Boot/Spring Cloud版本对bootstrap或Config Import的接入方式不同，不直接把历史加载方式用于新版本。


## 10. 镜像、容器、仓库与版本

| 对象 | 含义 |
|---|---|
| 可执行JAR | Spring Boot打包后的应用制品 |
| Docker镜像 | 包含运行环境和应用的只读模板 |
| 容器 | 从镜像启动的运行实例，具有自己的进程和可写层 |
| Registry | 存储与分发镜像的服务 |
| Repository/tag | 镜像名称空间与标签，tag可能被移动 |
| Digest | 对镜像内容的校验标识，用于精确锁定发布制品 |

课程镜像命名使用`docker_storage/<project.artifactId>`，运行脚本又使用`docker_storage/$JOB_NAME`。这要求Jenkins任务名与artifactId一致；若不一致，运行会拉错/找不到镜像。

工程补充：把服务名与Jenkins任务名分开显式定义，使用`<REGISTRY_HOST>/docker_storage/<SERVICE_NAME>:<COMMIT_OR_BUILD_ID>`，发布记录绑定digest，避免无标签命令隐含使用latest。

测试和生产应尽量晋级同一已验证镜像，环境通过运行配置注入；不要测试时构建一次、上线时从另一个分支重新构建，却仍认为是同一制品。


## 11. POM：可执行JAR与镜像插件

讲义以user服务为例：

- `finalName`为`heima-leadnews-user`；
- `spring-boot-maven-plugin`执行`repackage`；
- 编译器插件示例版本`3.7.0`，编译目标Java8；
- `com.spotify:dockerfile-maven-plugin:1.3.6`构建镜像；
- repository拼接`${docker.image}/${project.artifactId}`；
- buildArgs把`JAR_FILE`设为`target/${project.build.finalName}.jar`。

关键连接：

~~~text
Maven finalName → target/<finalName>.jar
                             ↓ buildArg JAR_FILE
Dockerfile COPY ${JAR_FILE} app.jar
                             ↓
镜像中 /app.jar → JVM启动
~~~

`repackage`确保得到包含运行入口和依赖的Spring Boot可执行JAR，不是只复制普通class。镜像插件也需要正确Dockerfile、构建上下文、Docker客户端/daemon访问权限。

该Spotify插件属于课程使用方式，现代项目可按环境选择受维护的Docker CLI/BuildKit、Jib或Spring Boot Buildpacks等方案。本文不虚构课程已经使用这些替代工具。


## 12. Dockerfile与JVM参数顺序

原Dockerfile使用`java:8`、`VOLUME /tmp`、`ARG JAR_FILE`、复制JAR、`JAVA_OPTS`和`PARAMS`，入口写成：

~~~text
java -jar $JAVA_OPTS /app.jar $PARAMS
~~~

**关键修正：** Java启动形式为`java [JVM选项] -jar <jar文件> [应用参数]`。把JAVA_OPTS放在`-jar`后会使第一个选项被当成JAR文件，导致启动失败。

保留课程变量模式的修正示意（基础镜像需替换）：

~~~dockerfile
FROM <SUPPORTED_JAVA_RUNTIME_IMAGE>
WORKDIR /
ARG JAR_FILE
COPY ${JAR_FILE} /app.jar
ENV JAVA_OPTS=""
ENV PARAMS=""
ENTRYPOINT ["sh", "-c", "exec java $JAVA_OPTS -jar /app.jar $PARAMS"]
~~~

`exec`让Java接收容器停止信号；变量展开和参数分词仍需受控，不能把不可信输入直接塞进Shell参数。该示意未运行，也不是完整生产Dockerfile。

补充：`java:8`应按支持状态选择基础镜像并锁定版本；Xmx只限制Java堆，不限制整个容器内存，Metaspace/线程/native也占内存；`VOLUME /tmp`通常创建卷，不保证数据永不丢失；长期数据和日志需明确挂载、轮转与备份。


## 13. 构建机本地部署：课程步骤

讲义先在Jenkins所在机器演示：检出 → Maven打包 → 构建镜像 → 删除旧容器 → 清理镜像 → 用prod profile启动新容器。

启动部分使用：

~~~text
docker run -d --net=host
  -e PARAMS="--spring.profiles.active=prod"
  --name <SERVICE_NAME> docker_storage/<SERVICE_NAME>
~~~

这是脱敏后的结构展示，不是本次执行命令。`--net=host`使用主机网络，服务端口直接占用宿主机端口，多个实例/服务容易冲突；Linux课程行为不能直接等同Windows或所有Docker Desktop环境。

`PARAMS`只有在Dockerfile入口读取该变量时才会成为Spring Boot参数；profile应在容器实际日志/Nacos配置中确认，不能只看Jenkins变量已填写。

原脚本用`docker ps -a -f name=$JOB_NAME`查容器再强制删除，名称过滤可能匹配多个近似名；有文件夹的Jenkins JOB_NAME也不一定合法作为Docker名称。工程应采用明确服务变量和精确对象标识。


## 14. 私有Registry与镜像生命周期

课程为了多服务器分发镜像，部署Docker Registry；示例将仓库数据目录挂载到宿主机，映射Registry服务端口，并用`/v2/_catalog`查看repository列表。

~~~text
构建服务器：build → tag → push
                           ↓ Registry（持久化镜像）
运行服务器：               pull → inspect → run
~~~

概念：

- `tag`为已有镜像增加名称，不重新编译应用；
- `push`把指定名称/版本的镜像上传仓库；
- `pull`拉取同一镜像到目标daemon；
- 删除本地tag不自动删除Registry内容；
- `_catalog`列出仓库不等于确认具体tag/digest已经发布成功。

讲义将HTTPS作为生产要求，也展示开发局域网的insecure-registry简化方式。本篇保留这个背景，实际方案使用TLS、认证和可信证书，不复制关闭保护的操作。Registry数据应持久化并备份，latest标签与无版本镜像不能支撑可靠回滚。

原文Docker服务重载命令拼为`daemon-reolad`，正确名称是`daemon-reload`；只记录修正，未重启任何服务。


## 15. 远程部署：构建端与运行端

讲义远程流程使用Jenkins任务参数`docker_registry`和`JOB_NAME`：

| 构建端 | 运行端 |
|---|---|
| Maven打包并构建镜像 | 接收授权SSH部署任务 |
| 为镜像增加Registry名称 | 从Registry拉取对应镜像 |
| 推送镜像 | 替换旧容器并以prod启动 |
| 清理本地tag | 检查目标镜像和容器 |

目标服务器参数需要通过插件明确传递或放入受控部署脚本。不能假设Jenkins本地定义的`$docker_registry`、`$JOB_NAME`在远程Shell里自动存在。

原脚本推送时使用`docker_storage/$JOB_NAME`，远程同名拉取，未给出不可变tag和digest校验。工程补充：显式记录完整IMAGE_REF、digest、service、profile、目标host和commit，把它们作为部署输入。

构建成功、推送成功、SSH成功、容器启动和应用健康是五个不同关口。某一步失败应使构建状态失败并保留上一步证据；不能忽略退出码继续删旧服务。


## 16. 旧容器替换、失败处理与回滚

课程脚本有直接强制删除旧容器、全局`docker image prune -f`、启动新容器的步骤。它便于演示，但可能造成停机和恢复困难。

工程补充的发布顺序：

~~~text
完成构建/测试 → 推送不可变版本 → 目标拉取并核对digest
      ↓
预检配置/端口/依赖 → 启动候选实例 → 等待就绪
      ↓
切换流量 → 验证关键业务 → 保留旧版本供回滚
      ↓
按独立保留策略清理历史镜像
~~~

单机单端口不能同时启动两个占相同端口的host网络容器。如果只能停旧再启新，就要明确维护窗口与恢复步骤；不要在同一配置下宣称零停机。

回滚应记录上一个镜像digest、配置版本、启动参数和数据库迁移状态。只把镜像tag切回去不保证数据库、Nacos配置和消息schema兼容。

不要在每次发布中无范围清理回滚镜像；使用保留天数、版本数量、引用和磁盘指标管理。上述部署改进是笔记方案，本次没有替换/删除任何容器。


## 17. 网关、Nginx与部署联调

讲义要求按user服务方式部署APP网关，并修改本地`heima-leadnews-app.conf`中的upstream，使页面请求进入已部署网关。

占位示意：

~~~nginx
upstream heima-app-gateway {
    server <APP_GATEWAY_HOST>:<APP_GATEWAY_PORT>;
}
~~~

前端访问 → Nginx → 网关 → Nacos发现的微服务 → 数据库/Redis/Kafka/搜索等依赖，是一条完整链路。容器列表中有名称不等于这条链路已经通。

工程联调检查：

1. 网关地址和实际部署主机一致；
2. 服务注册的是可从网关访问的地址，不是容器内部不可达地址；
3. prod profile加载正确namespace/group/Data ID；
4. 数据库、Redis、Kafka、ES、MinIO等地址可由运行环境访问；
5. 页面登录/文章列表/详情/搜索/行为请求正常，日志与响应对应；
6. 健康检查端点权限与就绪标准明确。

讲义在本地部署和远程部署示例间切换主机，最后联调用的是本地示例网关地址。实际跟随远程发布时，要重新核对，不能照搬截图里的IP。


## 18. 四种构建触发方式

| 触发器 | 课程介绍 | 适用边界 |
|---|---|---|
| 远程URL触发 | 带构建Token的job/build路径 | 需身份、权限和正确HTTP方式 |
| 其他工程完成后触发 | 上游工程完成触发下游 | 基础依赖构建后启动微服务构建等 |
| Build periodically | 到时间就构建 | 不依赖源码是否有变化 |
| Poll SCM | 定时检查源码是否变化，有变化才构建 | 有扫描开销，频率要合理 |

讲义展示URL中的固定Token，本篇不复现。现代Jenkins构建API通常用经过认证的POST，并按接口/认证方式处理CSRF保护，不能把示例GET链接或裸Token当作通用方案。

“上游任务结束”也不等于成功；链式任务应只在满足质量结果时继续。多个服务依赖同一父版本时，要明确上下游使用相同commit/依赖版本。

工程补充：Git Webhook事件触发可以减少轮询，但本篇不声称课程已配置Webhook，也没有向Git平台创建回调。


## 19. Jenkins Cron与H的含义

Jenkins定时表达式是五列：**分 时 日 月 周**。不要照搬第10天XXL-JOB的六列表达式`0 0 2 * * ?`。

| 字段 | 取值 |
|---|---|
| 分 | 0～59 |
| 时 | 0～23 |
| 日 | 1～31 |
| 月 | 1～12 |
| 周 | 0～7，0/7为星期日 |

课程例子：

~~~text
H/30 * * * *      每小时约每30分钟触发一次
H H/2 * * *       每两小时的分散时间触发
0 8,12,22 * * *   每天8:00、12:00、22:00
H 12 * * *        每天12点这一小时内的固定散列分钟
H 18 * * *        每天18点这一小时内的固定散列分钟
~~~

讲义说H是随机数，更准确是基于任务名的稳定散列值，用来把任务分散到不同分钟/小时。它通常不会每次执行都重新随机；重命名任务可能改变散列时间。

`H 12 * * *`不是精确12:00；若需要准点，用相应固定分钟并考虑负载。时区也应明确，不能默认所有构建节点的时钟配置相同。


## 20. Poll SCM、任务链与可追踪流水线

讲义认为Poll SCM增加扫描开销，不推荐频繁使用。它检查所配置的SCM是否变化，不是“每次都重新构建”；与到点即构建的Build periodically不同。

工程选择：小团队低频轮询可用；频繁提交可采用Webhook或多分支集成。无论触发方式，都要让一份构建记录回答：哪个commit、哪些测试、哪份JAR、哪个镜像digest、哪个环境、是否健康、如何回滚。

讲义主要展示UI任务配置，以下是把CI步骤放入版本控制的**Pipeline补充示意**，没有包含真实发布和凭据：

~~~groovy
pipeline {
    agent any
    stages {
        stage('Checkout') {
            steps { checkout scm }
        }
        stage('Build and Test') {
            steps { sh 'mvn -B clean verify' }
        }
    }
    post {
        always {
            junit testResults: '**/target/surefire-reports/*.xml',
                  allowEmptyResults: true
        }
    }
}
~~~

示意需项目提供Maven、正确POM和测试配置；它不是完整黑马头条生产Jenkinsfile。实际应按项目测试要求防止“没有报告”被视为质量通过，再增加镜像构建、制品归档和授权环境发布阶段。


## 21. 讲义操作与代码修正表

| 原文现象 | 原因/风险 | 笔记中的处理 |
|---|---|---|
| 旧Jenkins/Git和java:8 | 与当前兼容/维护状态不同 | 标明课程版本，按支持矩阵选工具 |
| `java -jar $JAVA_OPTS /app.jar` | JVM参数被当作JAR位置 | JVM参数放在-jar之前，应用参数放在JAR之后 |
| `-Dmaven.test.skip=true` | 不构成测试通过证据 | 解释跳过编译/运行，CI有独立验证关口 |
| dockerfile:build称为构建容器 | 混淆制品与运行实例 | 明确build镜像、run容器 |
| JOB_NAME拼镜像和容器名 | 任务名不一定等于artifactId或合法服务名 | 显式SERVICE_NAME/IMAGE_REF |
| 无tag镜像 | 隐含latest，不可追踪 | commit/build tag与digest |
| name过滤后强制删所有结果 | 可能匹配近似对象 | 精确对象名与可恢复替换 |
| 先删旧容器再启动 | 新镜像或配置失败就停服 | 拉取预检、就绪验证、回滚计划 |
| 每次全局prune | 回滚镜像/其他任务资源受影响 | 独立有范围的保留清理策略 |
| root、chmod 777、关闭证书校验 | 权限或信任边界过宽 | 最小权限、可信证书、隔离Agent |
| 更新源用HTTP / insecure-registry | 传输与认证保护不足 | 实际环境用TLS与认证 |
| `daemon-reolad` | 命令拼写错误 | `daemon-reload` |
| `spring.profile.active`文字拼写 | profile键错误 | 正确`spring.profiles.active` |
| 远程变量无显式传递 | 本地变量未必存在远端 | 受控部署参数/脚本 |
| 容器创建即认为部署成功 | 应用可能启动失败/配置错误 | 就绪、业务链路和回滚验证 |

这些是对可见讲义片段的修正建议，不代表已经修改或执行参考工程。


## 22. 部署验收清单（本次未执行）

### 构建与制品

1. Git记录实际commit，Maven依赖可在干净构建环境解析；
2. 父模块、公共模块和目标微服务使用相同源码版本；
3. 测试执行并生成报告，不只看到“跳过测试”；
4. JAR可执行，COPY路径正确，Dockerfile参数顺序正确；
5. 镜像包含目标版本并通过启动/必要检查；
6. Registry推送成功，目标拉取digest一致。

### 运行与业务

1. profile、Nacos namespace/group/Data ID一致；
2. 端口、网络、服务注册和依赖连接正确；
3. 容器停止信号、重启策略、日志与资源配额符合设计；
4. 新实例就绪，网关能路由到它；
5. 页面登录、文章详情、搜索和行为接口完成适当冒烟验证；
6. 消息消费者/任务执行器正常注册，不因新副本错误重复消费/调度；
7. 演练配置错误、推送/拉取失败、启动失败和回滚；
8. 旧镜像与配置记录保留，数据库/消息兼容性明确。

发布结果应包括commit、构建号、测试结果、镜像引用/digest、配置版本、环境、健康验证和回滚目标。本次只创建和校验学习笔记，没有执行上述构建、部署或测试。


## 23. 故障排查路线

| 现象 | 优先检查 |
|---|---|
| Git拉取失败 | 仓库URL、凭据权限、分支、网络/TLS |
| POM或父依赖找不到 | 工作区层级、-f路径、父模块、Maven仓库配置 |
| 本机能构建Jenkins不能 | JDK/Maven版本、依赖、环境变量、Agent权限 |
| Docker build失败 | JAR存在、构建上下文、插件/daemon连接 |
| 启动提示无法访问JAR | -jar后参数位置、COPY路径、文件权限 |
| push/pull失败 | Registry名称、tag、认证/TLS、目标网络 |
| 找不到镜像 | artifactId与JOB_NAME不一致、tag/digest不匹配 |
| 容器立刻退出 | JVM参数、资源、profile、Nacos/依赖连接 |
| 服务未注册或不可达 | discovery地址、注册IP、namespace、网络、端口 |
| 页面502/超时 | Nginx upstream、网关启动、路由和服务健康 |
| prod读到dev配置 | 启动参数是否生效，正确profile/Data ID/namespace |
| 远程变量为空 | Jenkins参数没有传给远端Shell |
| 每次都定时构建 | 配的是Build periodically，而非Poll SCM |

排查按Git → Maven → 镜像 → Registry → 容器 → 配置/发现 → 网关 → 页面逐层进行，不要靠反复删除镜像、重装软件掩盖第一处错误。保留构建号和关键日志，凭据不输出到控制台。


## 24. 复习与自测

~~~text
Git固定commit → Jenkins检出 → Maven测试/打包 → Docker镜像
                                      ↓
Registry不可变版本 → 目标拉取 → prod配置 → 启动与就绪
                                      ↓
Nginx/网关业务验证 → 观测/回滚 → 下一次迭代
~~~

1. **CI成功等于已经部署吗？** 不等于，它首先说明集成验证结果，部署还有环境和健康关口。
2. **install有什么作用？** 把构建制品安装到本地Maven仓库，供其他模块解析；不是启动服务。
3. **dockerfile:build生成什么？** 镜像，容器需另行run。
4. **JVM参数应放在哪？** -jar之前；JAR之后是应用参数。
5. **JAVA_OPTS和PARAMS如何生效？** 由镜像入口展开，不能只定义变量就认为Java自动使用。
6. **为什么Jenkins JDK与项目JDK要区分？** Jenkins运行和应用编译/运行有各自版本兼容要求。
7. **Data ID怎么拼？** prefix-profile.file-extension；profile为空则不带中间连字符。
8. **为什么latest不利于回滚？** tag会移动，无法仅凭名称确定已经验证的内容，应记录digest。
9. **本地JOB_NAME会自动出现在远端吗？** 不保证，部署插件/脚本需要明确传参。
10. **H每次会随机换分钟吗？** 通常是任务名稳定散列，用于分散负载，而非每次重新随机。
11. **第10天Cron能直接粘到Jenkins吗？** 不能，课程XXL-JOB是六列，Jenkins一般五列。
12. **有容器就算成功上线吗？** 不算，要验证就绪、配置、网关与关键业务。

### 资料覆盖

| 已读取资料 | 内容 |
|---|---|
| 项目部署_持续集成.md | CI、SDLC/敏捷、Jenkins工具与插件、Maven/Git/Docker、Nacos环境、私有仓库、远程部署、联调和触发器 |

已查看配套文件清单，但没有读取`.gitignore`/SQL正文、解压安装包或运行部署脚本。本文的Pipeline、不可变镜像、健康检查、回滚和权限修正为已标注的工程补充，文件校验与业务部署验证分别记录。
